# Model Evaluation - Movie Recommendation

Notebook ini digunakan untuk menjelaskan evaluasi model rekomendasi. Proses evaluasi utama dijalankan lewat script agar bisa diulang dengan stabil, lalu notebook ini membaca hasil CSV untuk presentasi.

## 1. Model Yang Dievaluasi

Model runtime MVP:

```text
PySpark ML Word2Vec + Qdrant
```

Jenis sistem:

```text
Content-based recommendation
```

Input model:

```text
movie_document_weighted
```

Output model:

```text
Vector film 64 dimensi
```

## 2. Peran Model Di Notebook Lama

| Model | Peran |
| --- | --- |
| CountVectorizer | Baseline sederhana berbasis frekuensi kata. |
| TF-IDF | Baseline teks yang lebih kuat untuk pembanding. |
| K-Means | Analisis segmentasi/cluster film. |
| Word2Vec | Model dense vector untuk Qdrant runtime. |

Untuk aplikasi MVP, runtime menggunakan Word2Vec + Qdrant karena Qdrant lebih cocok untuk dense vector yang kecil dan cepat dicari.

## 3. Cara Menjalankan Evaluasi

Jalankan dari root project:

```bash
docker compose up -d qdrant
docker compose run --rm ml python ml/scripts/modelling/evaluate_recommendations.py
```

Output:

```text
ml/reports/modelling/recommendation_examples.csv
ml/reports/modelling/model_evaluation_summary.csv
```

In [ ]:
from pathlib import Path
import pandas as pd

def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'ml').exists() and (candidate / 'docker-compose.yml').exists():
            return candidate
    raise RuntimeError('Project root tidak ditemukan')

PROJECT_ROOT = find_project_root(Path.cwd())
REPORT_DIR = PROJECT_ROOT / 'ml' / 'reports' / 'modelling'
REPORT_DIR

: 

## 4. Ringkasan Word2Vec

Ringkasan training Word2Vec dibaca dari `word2vec_summary.csv`.

In [ ]:
word2vec_summary = pd.read_csv(REPORT_DIR / 'word2vec_summary.csv')
word2vec_summary

## 5. Summary Evaluasi

Metric yang digunakan:

- `recommendation_count`: jumlah rekomendasi yang dikembalikan.
- `avg_similarity_score`: rata-rata similarity dari Qdrant.
- `avg_genre_overlap`: rata-rata irisan genre antara film acuan dan rekomendasi.
- `avg_vote_average`: rata-rata rating rekomendasi.
- `avg_vote_count`: rata-rata jumlah vote rekomendasi.

In [ ]:
summary_path = REPORT_DIR / 'model_evaluation_summary.csv'
if summary_path.exists():
    evaluation_summary = pd.read_csv(summary_path)
    display(evaluation_summary)
else:
    print('File evaluasi belum ada. Jalankan script evaluate_recommendations.py terlebih dahulu.')

## 6. Contoh Hasil Rekomendasi

Tabel ini menunjukkan hasil rekomendasi per query.

In [ ]:
examples_path = REPORT_DIR / 'recommendation_examples.csv'
if examples_path.exists():
    examples = pd.read_csv(examples_path)
    display(examples.head(30))
else:
    print('File recommendation_examples.csv belum ada. Jalankan script evaluate_recommendations.py terlebih dahulu.')

## 7. Melihat Satu Query

Gunakan cell berikut untuk melihat rekomendasi dari satu film tertentu.

In [ ]:
query = 'Interstellar'

if examples_path.exists():
    examples = pd.read_csv(examples_path)
    cols = [
        'rank',
        'recommended_title',
        'recommended_release_year',
        'recommended_genres',
        'similarity_score',
        'genre_overlap',
        'vote_average',
    ]
    display(examples[examples['query'] == query][cols])
else:
    print('File recommendation_examples.csv belum ada.')

## 8. Kesimpulan Evaluasi

Model Word2Vec + Qdrant sudah dapat menghasilkan rekomendasi berbasis kemiripan konten film. Untuk MVP, pendekatan ini cukup karena tidak membutuhkan data user dan dapat langsung digunakan oleh backend.

Kelebihan:

- cocok untuk dataset tanpa user history,
- vector kecil dan praktis untuk Qdrant,
- search cepat setelah indexing,
- mudah dijelaskan sebagai content-based recommendation.

Keterbatasan:

- belum personal untuk tiap user,
- query natural language masih perlu intent parsing,
- hybrid ranking di backend masih perlu ditambahkan.